In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [42]:
train = pd.read_csv("/Users/rahulmanchanda/Desktop/Machine Learning -Credit Card Fraud Detection /data/archive (5)/fraudTrain.csv")
test = pd.read_csv("/Users/rahulmanchanda/Desktop/Machine Learning -Credit Card Fraud Detection /data/archive (5)/fraudTest.csv")



print("Training Data Shape:", train.shape)
print("Testing Data Shape:", test.shape)

train.head()

Training Data Shape: (1296675, 23)
Testing Data Shape: (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [43]:
print("Missing Values in Training Data:")
train.isnull().sum()[train.isnull().sum() > 0]



Missing Values in Training Data:


Series([], dtype: int64)

In [44]:
print("Missing Values in Testing Data:")
test.isnull().sum()[test.isnull().sum() > 0]

Missing Values in Testing Data:


Series([], dtype: int64)

In [45]:
print("Duplicate Records in Training Data:", train.duplicated().sum())
print("Duplicate Records in Testing Data:", test.duplicated().sum())

Duplicate Records in Training Data: 0
Duplicate Records in Testing Data: 0


In [46]:
def prepare_features(data):

    data = data.copy()

    # Convert date columns
    data["trans_date_trans_time"] = pd.to_datetime(data["trans_date_trans_time"])
    data["dob"] = pd.to_datetime(data["dob"])

    # Extract transaction date features
    data["hour"] = data["trans_date_trans_time"].dt.hour
    data["day"] = data["trans_date_trans_time"].dt.day
    data["month"] = data["trans_date_trans_time"].dt.month
    data["day_of_week"] = data["trans_date_trans_time"].dt.dayofweek

    # Calculate customer age at transaction time
    data["age"] = (
        (data["trans_date_trans_time"] - data["dob"]).dt.days / 365.25
    ).astype(int)

    # Calculate distance between customer and merchant
    lat1 = np.radians(data["lat"])
    lon1 = np.radians(data["long"])
    lat2 = np.radians(data["merch_lat"])
    lon2 = np.radians(data["merch_long"])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0, 1)

    data["customer_merchant_distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

    # Remove unnecessary columns
    drop_cols = [
        "Unnamed: 0",
        "cc_num",
        "first",
        "last",
        "street",
        "city",
        "zip",
        "trans_date_trans_time",
        "dob",
        "trans_num",
        "unix_time",
        "lat",
        "long",
        "merch_lat",
        "merch_long"
    ]

    data.drop(columns=drop_cols, inplace=True)

    return data

In [47]:
train = prepare_features(train)
test = prepare_features(test)

print("Training Data Shape:", train.shape)
print("Testing Data Shape:", test.shape)

display(train.head())

Training Data Shape: (1296675, 14)
Testing Data Shape: (555719, 14)


,merchant,category,amt,gender,state,city_pop,job,is_fraud,hour,day,month,day_of_week,age,customer_merchant_distance_km
0,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,NC,3495,"Psychologist, counselling",0,0,1,1,1,30,78.597568
1,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,WA,149,Special educational needs teacher,0,0,1,1,1,40,30.212176
2,fraud_Lind-Buckridge,entertainment,220.11,M,ID,4154,Nature conservation officer,0,0,1,1,1,56,108.206083
3,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,MT,1939,Patent attorney,0,0,1,1,1,51,95.673231
4,fraud_Keeling-Crist,misc_pos,41.96,M,VA,99,Dance movement psychotherapist,0,0,1,1,1,32,77.556744


In [48]:
TARGET = "is_fraud"

X = train.drop(columns=[TARGET])
y = train[TARGET]

X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("Test Features Shape:", X_test.shape)
print("Test Target Shape:", y_test.shape)

Features Shape: (1296675, 13)
Target Shape: (1296675,)
Test Features Shape: (555719, 13)
Test Target Shape: (555719,)


In [49]:
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

Numerical Columns: ['amt', 'city_pop', 'age', 'customer_merchant_distance_km']
Categorical Columns: ['merchant', 'category', 'gender', 'state', 'job']


In [50]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Validation Features:", X_val.shape)

Training Features: (1037340, 13)
Validation Features: (259335, 13)


In [51]:
# There is no null value so we are not using any imputation in numerical and categorical pipeline
preprocessor = ColumnTransformer([
    ("numerical", StandardScaler(), numerical_cols),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [52]:

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Processed Training Data Shape:", X_train_processed.shape)
print("Processed Validation Data Shape:", X_val_processed.shape)
print("Processed Testing Data Shape:", X_test_processed.shape)

Processed Training Data Shape: (1037340, 1258)
Processed Validation Data Shape: (259335, 1258)
Processed Testing Data Shape: (555719, 1258)


In [53]:
print("Training Data Ready:", X_train_processed.shape)
print("Validation Data Ready:", X_val_processed.shape)
print("Testing Data Ready:", X_test_processed.shape)

print("\nTraining Target Distribution:")
display(y_train.value_counts())

print("\nValidation Target Distribution:")
display(y_val.value_counts())

Training Data Ready: (1037340, 1258)
Validation Data Ready: (259335, 1258)
Testing Data Ready: (555719, 1258)

Training Target Distribution:


is_fraud
0    1031335
1       6005
Name: count, dtype: int64


Validation Target Distribution:


is_fraud
0    257834
1      1501
Name: count, dtype: int64

In [55]:
print("\nTesting Target Distribution:")
display(y_test.value_counts())


Testing Target Distribution:


is_fraud
0    553574
1      2145
Name: count, dtype: int64

In [56]:
from scipy.sparse import save_npz
import joblib
import os

# Create folder
os.makedirs("processed_data", exist_ok=True)

# Save processed features
save_npz("processed_data/X_train_processed.npz", X_train_processed)
save_npz("processed_data/X_val_processed.npz", X_val_processed)
save_npz("processed_data/X_test_processed.npz", X_test_processed)

# Save target variables
y_train.to_csv("processed_data/y_train.csv", index=False)
y_val.to_csv("processed_data/y_val.csv", index=False)
y_test.to_csv("processed_data/y_test.csv", index=False)

# Save fitted preprocessor
joblib.dump(preprocessor, "processed_data/preprocessor.pkl")

print("All processed datasets saved successfully!")

All processed datasets saved successfully!
